# STAT 940 Data Challenge 2: Ensemble of Fine-Tuned Transformers for Text Regression

Fine-tunes four pretrained language models from Hugging Face (BERT, DistilBERT, ALBERT and GPT-2) to predict a
numeric score from text, then averages their predictions.
Competition: [uwstat940f23dc2 on Kaggle](https://www.kaggle.com/competitions/uwstat940f23dc2).

Runs locally on Linux with a CUDA GPU.

## 1. Download the Data

In [2]:
import os

if not os.path.isdir("data"):
    !kaggle competitions download -c uwstat940f23dc2
    !unzip -q uwstat940f23dc2.zip -d data

## 2. Imports and Settings

In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, get_linear_schedule_with_warmup, set_seed)
from transformers.utils import logging as hf_logging

SEED = 42
set_seed(SEED)
hf_logging.set_verbosity_error()

DATA_DIR = "data"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"

MODELS = {
    "BERT":       {"checkpoint": "bert-base-uncased",       "lr": 2e-5},
    "DistilBERT": {"checkpoint": "distilbert-base-uncased", "lr": 3e-5},
    "ALBERT":     {"checkpoint": "albert-base-v2",          "lr": 2e-5},
    "GPT-2":      {"checkpoint": "gpt2",                    "lr": 3e-5},
}

MAX_LENGTH = 512
EPOCHS = 4
BATCH_SIZE = 16
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.1

/home/u-waterloo/Documents/STAT-940---Deep-Learning-Data-Challenges-/Data Challenge 2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Load the Data

The first column of `train.csv` is the text and the second is the score. The test texts are read from `test.csv` if it exists, otherwise from `sample_submission.csv`. Scores are standardized for training and converted back for predictions.

In [4]:
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_path = f"{DATA_DIR}/test.csv" if os.path.exists(f"{DATA_DIR}/test.csv") else f"{DATA_DIR}/sample_submission.csv"
test_df = pd.read_csv(test_path)

TEXT_COL, TARGET_COL = train_df.columns[0], train_df.columns[1]
test_text_col = TEXT_COL if TEXT_COL in test_df.columns else test_df.columns[0]

texts = train_df[TEXT_COL].astype(str).tolist()
scores = train_df[TARGET_COL].to_numpy(dtype=np.float32)
test_texts = test_df[test_text_col].astype(str).tolist()

train_texts, valid_texts, y_train, y_valid = train_test_split(
    texts, scores, test_size=0.1, random_state=SEED
)
y_mean, y_std = y_train.mean(), y_train.std()

print(f"{len(train_texts)} training, {len(valid_texts)} validation, {len(test_texts)} test texts")
print(f"Score: mean {y_mean:.3f}, std {y_std:.3f}")

2700 training, 300 validation, 500 test texts
Score: mean 3.486, std 0.657


## 4. Dataset

In [5]:
class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, targets=None):
        self.encodings = tokenizer(texts, truncation=True, max_length=MAX_LENGTH)
        self.targets = targets

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, i):
        item = {key: values[i] for key, values in self.encodings.items()}
        if self.targets is not None:
            item["labels"] = float((self.targets[i] - y_mean) / y_std)
        return item


def make_loader(texts, tokenizer, targets=None, shuffle=False):
    return DataLoader(TextDataset(texts, tokenizer, targets), batch_size=BATCH_SIZE, shuffle=shuffle,
                      collate_fn=DataCollatorWithPadding(tokenizer))

## 5. Fine-Tuning

Each model gets a single-output regression head and is fine-tuned end to end with AdamW, a linear learning-rate schedule with warm-up, gradient clipping and bfloat16 mixed precision. The epoch with the lowest validation RMSE is kept.

In [6]:
def rmse(predictions, targets):
    return float(np.sqrt(np.mean((predictions - targets) ** 2)))


@torch.no_grad()
def predict(model, loader):
    model.eval()
    outputs = []
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
        with torch.autocast(device_type=DEVICE, dtype=torch.bfloat16, enabled=USE_AMP):
            logits = model(**batch).logits
        outputs.append(logits.float().squeeze(-1).cpu())
    return torch.cat(outputs).numpy() * y_std + y_mean


def fine_tune(name, checkpoint, lr):
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint, num_labels=1, ignore_mismatched_sizes=True
    )
    model.config.pad_token_id = tokenizer.pad_token_id
    model.to(DEVICE)

    train_loader = make_loader(train_texts, tokenizer, y_train, shuffle=True)
    valid_loader = make_loader(valid_texts, tokenizer)
    test_loader = make_loader(test_texts, tokenizer)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    total_steps = EPOCHS * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_FRACTION * total_steps), total_steps)

    best_rmse, best_path = float("inf"), f"best_{name.replace('/', '_')}.pt"
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for batch in train_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            labels = batch.pop("labels").float()
            with torch.autocast(device_type=DEVICE, dtype=torch.bfloat16, enabled=USE_AMP):
                logits = model(**batch).logits.squeeze(-1)
            loss = F.mse_loss(logits.float(), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        valid_rmse = rmse(predict(model, valid_loader), y_valid)
        print(f"{name} | epoch {epoch}/{EPOCHS} | validation RMSE {valid_rmse:.4f}")
        if valid_rmse < best_rmse:
            best_rmse = valid_rmse
            torch.save(model.state_dict(), best_path)

    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    valid_preds, test_preds = predict(model, valid_loader), predict(model, test_loader)

    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return valid_preds, test_preds

## 6. Train the Models

Validation and test predictions are saved to disk.

In [7]:
valid_preds, test_preds = {}, {}
for name, cfg in MODELS.items():
    valid_preds[name], test_preds[name] = fine_tune(name, cfg["checkpoint"], cfg["lr"])
    np.save(f"valid_preds_{name}.npy", valid_preds[name])
    np.save(f"test_preds_{name}.npy", test_preds[name])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14976.70it/s]


BERT | epoch 1/4 | validation RMSE 0.4043
BERT | epoch 2/4 | validation RMSE 0.3790
BERT | epoch 3/4 | validation RMSE 0.3555
BERT | epoch 4/4 | validation RMSE 0.3598


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 13072.48it/s]


DistilBERT | epoch 1/4 | validation RMSE 0.3665
DistilBERT | epoch 2/4 | validation RMSE 0.3595
DistilBERT | epoch 3/4 | validation RMSE 0.3623
DistilBERT | epoch 4/4 | validation RMSE 0.3639


Loading weights: 100%|██████████| 25/25 [00:00<00:00, 46707.17it/s]


ALBERT | epoch 1/4 | validation RMSE 0.3459
ALBERT | epoch 2/4 | validation RMSE 0.3346
ALBERT | epoch 3/4 | validation RMSE 0.3636
ALBERT | epoch 4/4 | validation RMSE 0.3500


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 9074.60it/s]


GPT-2 | epoch 1/4 | validation RMSE 0.5157
GPT-2 | epoch 2/4 | validation RMSE 0.4224
GPT-2 | epoch 3/4 | validation RMSE 0.3691
GPT-2 | epoch 4/4 | validation RMSE 0.3851


## 7. Ensemble

The models' predictions are averaged. The table compares each model with the ensemble, using a constant prediction (the training mean) as a baseline.

In [8]:
ensemble_valid = np.mean(list(valid_preds.values()), axis=0)
ensemble_test = np.mean(list(test_preds.values()), axis=0)

results = {"Baseline (predict the mean)": rmse(np.full_like(y_valid, y_mean), y_valid)}
results.update({name: rmse(p, y_valid) for name, p in valid_preds.items()})
results["Ensemble (Average)"] = rmse(ensemble_valid, y_valid)

results_table = pd.DataFrame({"Validation RMSE": results}).round(4)
results_table

,Validation RMSE
Baseline (predict the mean),0.6536
BERT,0.3555
DistilBERT,0.3595
ALBERT,0.3346
GPT-2,0.3691
Ensemble (Average),0.3368


## 8. Write the Submission

In [9]:
submission = test_df.copy()
submission[TARGET_COL if TARGET_COL in test_df.columns else "score"] = ensemble_test
submission.to_csv("predictions_ensemble.csv", index=False)
submission.head()

,text,score
0,"Dear Local Newspaper, @CAPS1 has come to my at...",3.943198
1,"Dear Newspaper, @CAPS1 you know that two out o...",4.182133
2,"Dear, @ORGANIZATION1. There has been lots of t...",3.256700
3,"Dear @ORGANIZATION1, A regular day for a perso...",4.212922
4,"Libraries are places of learning. Books, magaz...",3.825494
